# morefs smoke test

Every example uses `morefs.memory.MemFS`, which needs the optional `pygtrie` package. A plain
`pip install morefs` does not install it; install the package with its `memfs` extra to run this notebook:
`pip install "morefs[memfs]"`.

## Example 1: Basic in-memory filesystem operations

In [ ]:
# Example 1: Basic in-memory filesystem operations
from morefs.memory import MemFS
import json

print("Example 1 - In-memory filesystem")

# Create in-memory filesystem
fs = MemFS()
print(f"Created: {type(fs).__name__}")

# Create directories and write files
fs.makedirs("project/src", exist_ok=True)
fs.makedirs("project/config", exist_ok=True)

config = {"app": "myapp", "version": "1.0"}
with fs.open("project/config/settings.json", "w") as f:
    json.dump(config, f)

with fs.open("project/src/main.py", "w") as f:
    f.write("print('Hello from morefs')")

# List and read
files = fs.ls("project/config", detail=False)
print(f"Config files: {files}")

with fs.open("project/config/settings.json", "r") as f:
    loaded = json.load(f)
print(f"Loaded config: {loaded}")
assert loaded == config
assert [p.rsplit("/", 1)[-1] for p in files] == ["settings.json"], files
with fs.open("project/src/main.py", "r") as f:
    assert f.read() == "print('Hello from morefs')"

print("Example 1 completed")

## Example 2: Dictionary-based filesystem

In [ ]:
# Example 2: Dictionary-based filesystem
from morefs.dict import DictFS
import json

print("Example 2 - Dictionary-based filesystem")

# Create dict filesystem
fs = DictFS()
print(f"Created: {type(fs).__name__}")

# Write multiple files
fs.makedirs("data", exist_ok=True)

data_files = {
    "data/users.json": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}],
    "data/config.json": {"debug": True, "port": 8080}
}

for path, content in data_files.items():
    with fs.open(path, "w") as f:
        json.dump(content, f)

# List and check
all_files = fs.find("data")
print(f"Files in data: {len(all_files)}")

# Read back
with fs.open("data/users.json", "r") as f:
    users = json.load(f)
print(f"Users: {len(users)}")
assert users == data_files["data/users.json"]
assert sorted(p.rsplit("/", 1)[-1] for p in all_files) == ["config.json", "users.json"], all_files

print("Example 2 completed")

## Example 3: File operations (copy, move, delete)

In [ ]:
# Example 3: File operations (copy, move, delete)
from morefs.memory import MemFS

print("Example 3 - File operations")

fs = MemFS()

# Create test files
fs.makedirs("temp", exist_ok=True)
with fs.open("temp/file1.txt", "w") as f:
    f.write("Test content 1")

with fs.open("temp/file2.txt", "w") as f:
    f.write("Test content 2")

print(f"Created files: {fs.ls('temp', detail=False)}")

# Copy file
fs.copy("temp/file1.txt", "temp/file1_copy.txt")
print(f"After copy: {len(fs.ls('temp', detail=False))} files")

# Move file
fs.move("temp/file2.txt", "temp/file2_moved.txt")
exists_old = fs.exists("temp/file2.txt")
exists_new = fs.exists("temp/file2_moved.txt")
print(f"Move: old exists={exists_old}, new exists={exists_new}")
assert not exists_old and exists_new
assert sorted(p.rsplit("/", 1)[-1] for p in fs.ls("temp", detail=False)) == ["file1.txt", "file1_copy.txt", "file2_moved.txt"]
assert fs.cat_file("temp/file1_copy.txt") == b"Test content 1"
assert fs.cat_file("temp/file2_moved.txt") == b"Test content 2"

print("Example 3 completed")

## Example 4: File metadata and information

In [ ]:
# Example 4: File metadata and information
from morefs.memory import MemFS

print("Example 4 - File metadata")

fs = MemFS()

# Create files with different sizes
fs.makedirs("files", exist_ok=True)

for i in range(3):
    content = "x" * (100 * (i + 1))
    with fs.open(f"files/file{i}.txt", "w") as f:
        f.write(content)

# Get file information
for i in range(3):
    path = f"files/file{i}.txt"
    info = fs.info(path)
    size = info['size']
    ftype = info['type']
    print(f"{path}: {size} bytes, type={ftype}")
    assert size == 100 * (i + 1) and ftype == "file", info

# Directory info
dir_info = fs.info("files")
print(f"Directory type: {dir_info['type']}")

# Check existence
print(f"Exists test: {fs.exists('files/file0.txt')}")
assert dir_info['type'] == "directory"
assert fs.exists('files/file0.txt') and not fs.exists('files/missing.txt')

print("Example 4 completed")

## Example 5: Finding and glob patterns

In [ ]:
# Example 5: Finding and glob patterns
from morefs.memory import MemFS

print("Example 5 - Finding and glob patterns")

fs = MemFS()

# Create directory structure
fs.makedirs("src/utils", exist_ok=True)
fs.makedirs("src/models", exist_ok=True)
fs.makedirs("tests", exist_ok=True)

# Create various files
files = {
    "src/main.py": "main",
    "src/utils/helper.py": "helper",
    "src/models/user.py": "user",
    "tests/test_main.py": "test",
    "README.md": "readme"
}

for path, content in files.items():
    with fs.open(path, "w") as f:
        f.write(content)

# Find all Python files
py_files = fs.glob("**/*.py")
print(f"Python files: {len(py_files)}")

# Find files in src
src_files = fs.find("src")
print(f"Files in src: {len(src_files)}")
assert sorted(p.rsplit("/", 1)[-1] for p in py_files) == ["helper.py", "main.py", "test_main.py", "user.py"], py_files
assert sorted(p.rsplit("/", 1)[-1] for p in src_files) == ["helper.py", "main.py", "user.py"], src_files

print("Example 5 completed")